# FIAP • Internet das Coisas & Telecomunicações
## Turma 5ECS • Ano 2026 | Checkpoint 4
**Título:** Plataformas IoT em Nuvem, Ingestão Massiva e Séries Temporais

> **Enunciado Original:**
> Dimensionar a arquitetura de ingestão de dados em nuvem para processamento de telemetria de 50.000 nós sensores transmitindo mensagens JSON de 128 bytes a cada 10 segundos. Dimensionar a capacidade do Message Broker (AWS IoT Core / Azure IoT Hub / EMQX), o particionamento do banco de séries temporais (InfluxDB / TimescaleDB) e a largura de banda de entrada requerida.

### Formulação Matemática Formal
$$\text{Taxa Mensagens} = \frac{N_{\text{dispositivos}}}{T_{\text{envio}}}, \quad \text{Throughput (MB/s)} = \frac{\text{Taxa Mensagens} \cdot \text{Tamanho (bytes)}}{10^6}$$


### Passo 1: Taxa Média de Ingestão de Mensagens
Com 50.000 dispositivos enviando a cada 10 segundos: Taxa = \frac{50.000}{10} = 5.000\text{ mensagens por segundo} (MPS).


### Passo 2: Throughput de Rede Requerido
Tamanho do payload de 128 bytes: Throughput = 5.000 \times 128\text{ B/s} = 640.000\text{ B/s} = 640\text{ KB/s} (\approx 5,12\text{ Mbps}). Largura de banda moderada e facilmente sustentável.


### Passo 3: Volume Diário de Armazenamento em Disco
Mensagens por dia: 5.000 \times 86.400 = 432.000.000\text{ mensagens/dia}. Volume bruto diário: 432\times 10^6 \times 128\text{ B} \approx 55,3\text{ GB/dia}.


### Passo 4: Otimização e Compressão em Banco Time-Series
Com bancos especializados em séries temporais (TimescaleDB / InfluxDB Gorilla Compression), a taxa de compressão típica de 85% reduz o volume armazenado para apenas 8,3 GB/dia.


### Passo 5: Estratégia de Particionamento (Chunking)
Particionamento de tabelas em chunks temporais de 24 horas indexados por Device_ID e Timestamp, permitindo expurgo automático (Data Retention Policy) após 90 dias sem travar o motor SQL.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

sensores = np.linspace(1000, 100000, 100)
mps = sensores / 10.0 # mensagens/segundo
vol_diario_gb = (mps * 86400 * 128) / (1024**3)
vol_comp_gb = vol_diario_gb * 0.15 # 85% compressao
fig, ax = plt.subplots(figsize=(6, 3.5), facecolor='#181436')
ax.set_facecolor('#100d22')
ax.plot(sensores / 1000, vol_diario_gb, color='#ef4444', lw=2, label='Volume Bruto sem Compressão (GB/dia)')
ax.plot(sensores / 1000, vol_comp_gb, color='#10b981', lw=2.5, label='Volume Comprimido Gorilla (GB/dia)')
ax.axvline(50, color='#8b5cf6', ls='--', label='Cenário Atual: 50k Dispositivos')
ax.set_title('Escala de Armazenamento Diário da Plataforma IoT', color='#ffffff', fontsize=11, fontweight='bold')
ax.set_xlabel('Quantidade de Dispositivos Conectados (Milhares)', color='#94a3b8', fontsize=9)
ax.set_ylabel('Armazenamento Diário Requerido (GB)', color='#94a3b8', fontsize=9)
ax.tick_params(colors='#94a3b8')
ax.grid(True, color='#3a3275', alpha=0.5, ls=':')
ax.legend(facecolor='#241f4d', edgecolor='#3a3275', labelcolor='#ffffff', fontsize=8)
plt.tight_layout()
plt.savefig('grafico_resolucao.png', dpi=200, facecolor=fig.get_facecolor())
plt.close()
